In [ ]:
import pedpy
import matplotlib.pyplot as plt
import pandas as pd
from shapely import from_wkt, Polygon
import json
from pedpy import compute_voronoi_density
from pedpy import compute_individual_voronoi_polygons
from pathlib import Path
from pedpy import (
    SpeedCalculation,
    compute_individual_speed,
)
from pedpy import FRAME_COL, ID_COL


In [ ]:
# Original exterior ring
exterior = [
    (-8.88, -11.1),
    (8.3, -11.1),
    (8.3, 27.95),
    (-8.88, 27.95),
    (-8.88, -11.1),
]

# Interior rings (excluding the door)
interior_rings = [
    # Left cutout
    [
        (-7, -11),
        (-3.57, -3),
        (-3.57, 19.57),
        (-1.52, 19.57),
        (-1.37, 19.71),
        (-0.87, 19.71),
        (-0.72, 19.57),
        (-0.42, 19.57),
        (-0.42, 21.23),
        (-0.72, 21.23),
        (-0.87, 21.09),
        (-1.37, 21.09),
        (-1.52, 21.23),
        (-1.67, 21.23),
        (-1.67, 21.18),
        (-1.545, 21.18),
        (-1.42, 21.065),
        (-1.42, 19.735),
        (-1.545, 19.62),
        (-3.62, 19.62),
        (-3.59, -3),
        (-7, -11),
    ],
    # Right cutout
    [
        (7, -11),
        (3.57, -3),
        (3.64, 19.64),
        (1.47, 19.57),
        (1.32, 19.71),
        (0.82, 19.71),
        (0.67, 19.57),
        (0.38, 19.57),
        (0.38, 21.23),
        (0.67, 21.23),
        (0.82, 21.09),
        (1.32, 21.09),
        (1.47, 21.23),
        (1.62, 21.23),
        (1.62, 21.18),
        (1.495, 21.18),
        (1.37, 21.065),
        (1.37, 19.735),
        (1.495, 19.62),
        (3.69, 19.69),
        (3.62, -3),
        (7, -11),
    ],
    # Bottom strip
    [(-6.8, -10.8), (6.8, -10.8), (6.8, -10.6), (-6.8, -10.6), (-6.8, -10.8)],
]

# Door coordinates
#door = [(-0.4, 19.57), (0.37, 19.57), (0.37, 19.3), (-0.4, 19.3), (-0.4, 19.57)]
# Create closed geometry (with door)
geometry = Polygon(exterior, interior_rings)

walkabel_area = pedpy.WalkableArea(geometry)
 
def read_json_file(file_path):
    with open(file_path, "r") as file:
        data = json.load(file)
    return data

inifile="files/inifile.json"
data = read_json_file(inifile)
measurement_line = pedpy.MeasurementLine(data["measurement_line"]["vertices"])
measurement_area = pedpy.MeasurementArea(data["measurement_area"]["vertices"])    
measurement_area = pedpy.MeasurementArea([[-0.72, 17], [0.67, 17], [0.67, 16], [-0.72, 16]])    
bounding = from_wkt(
    "POLYGON ((-8 -7, 8 -7, 8 20, -8 20, -8 -7))"
)
bounding_area = pedpy.WalkableArea(bounding)
#pedpy.plot_measurement_setup(traj=traj, walkable_area=walkabel_area, measurement_lines=[measurement_line], measurement_areas=[measurement_area]).axis('equal') 


In [ ]:
experiment_files = [
    "../trajectories_croma/1C070_cam6_cam5_frameshift0_Combined.txt",  # low motivation
    #"../trajectories_croma/2C070_cam6_cam5_frameshift0_Combined.txt",  # high motivation
]  
date = "2025-04-02_12-26-06"
simulation_files = [
    Path(f"files/variations/m1C070_base_{date}.sqlite")    
]

fps_values = {
    "experiment": 50,
    "simulation": 20
}

results = {}

def process_trajectory_file(filename, fps, file_type):
    """Loads, processes, and computes Voronoi density for a given trajectory file."""
    label = f"{file_type}_{Path(filename).stem}"  # Label based on file type

    if file_type == "experiment":
        df = pd.read_csv(filename, sep="\t", names=["id", "frame", "x", "y", "z", "m"], comment="#")
        traj = pedpy.TrajectoryData(df, frame_rate=fps)
        # 5016
        print(filename, traj.frame_rate)
    elif file_type == "simulation":
        traj = pedpy.load_trajectory_from_jupedsim_sqlite(filename)
        print(filename, traj.frame_rate)
     
    fps = traj.frame_rate   
    # Compute Voronoi density
    individual = compute_individual_voronoi_polygons(traj_data=traj, walkable_area=bounding_area)
    density_voronoi, intersecting = compute_voronoi_density(
        individual_voronoi_data=individual, measurement_area=measurement_area
    )
    individual_speed = compute_individual_speed(
        traj_data=traj,
        frame_step=5,
        speed_calculation=SpeedCalculation.BORDER_SINGLE_SIDED,
    )
    

    profile_data = individual_speed.merge(individual, on=[ID_COL, FRAME_COL])
    profile_data = profile_data.merge(traj.data, on=[ID_COL, FRAME_COL])

    # Store results
    results[label] = {
        "profile_data": profile_data,
        "density": density_voronoi,
        "fps": fps
    }

# Process all files
for filename in experiment_files:
    process_trajectory_file(filename, fps_values["experiment"], "experiment")

for filename in simulation_files:
    process_trajectory_file(filename, fps_values["simulation"], "simulation")

# Plot all densities
plt.figure(figsize=(10, 5))
for label, data in results.items():
    time = data["density"].index/ data["fps"]  # Convert frame index to time
    plt.plot(time, data["density"]["density"], label=label)

plt.xlabel("Time (seconds)")
plt.ylabel(r"$\rho$ (1/m²)")
plt.legend()
plt.title("Density vs Time for Experiments and Simulations")
plt.show()


In [ ]:
from pedpy import (
    get_grid_cells, plot_profiles
)
from pedpy import DensityMethod, compute_density_profile


grid_size = 0.4
grid_cells, _, _ = get_grid_cells(
    walkable_area=walkabel_area, grid_size=grid_size
)
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, layout="constrained")
for (label, data), ax in zip(results.items(), [ax1, ax2]):
    gaussian_density_profile = compute_density_profile(
        data=data["profile_data"],
        walkable_area=walkabel_area,
        grid_size=grid_size,
        density_method=DensityMethod.GAUSSIAN,
        gaussian_width=0.5,
    )
    cm = plot_profiles(
        walkable_area=walkabel_area,
        profiles=gaussian_density_profile,
        axes=ax,
        label="$\\rho$ / 1/$m^2$",
        vmin=0,
        vmax=3,
        title=label.split("_")[0],
    )

In [ ]:
#experiment_file = "../trajectories_croma/1C070_cam6_cam5_frameshift0_Combined.txt"  # low motivation
experiment_file = "../trajectories_croma/2C070_cam6_cam5_frameshift0_Combined.txt"  # high motivation
fps = 50
df = pd.read_csv(experiment_file, sep="\t", names=["id", "frame", "x", "y", "z", "m"], comment="#")
traj = pedpy.TrajectoryData(df, frame_rate=fps)
# 1) Filter rows where y >= 20
df_crossed = df[df["y"] >= 20].copy()
crossing_frames = df_crossed.groupby("id")["frame"].min().rename("crossing_frame")
crossing_frames_sorted = crossing_frames.sort_values()
crossing_order = pd.Series(range(1, len(crossing_frames_sorted) + 1),
                           index=crossing_frames_sorted.index,
                           name="crossing_order")
# crossing_order is a Series: index = pedestrian id, value = "1" for first to cross, "2" for second, etc.
crossing_info = pd.concat([crossing_frames, crossing_order], axis=1)
# crossing_info now has columns [crossing_frame, crossing_order], indexed by id

In [ ]:
individual = compute_individual_voronoi_polygons(traj_data=traj, walkable_area=bounding_area)
# Group by frame and compute mean
density_over_time = (individual
                     .groupby("frame")["density"]
                     .mean()
                     .rename("mean_individual_density"))

density_per_agent = (individual
                     .groupby("id")["density"]
                     .mean()
                     .rename("mean_density_per_agent"))

In [ ]:
# Convert to DataFrame (optional but convenient)
df_order = crossing_order.to_frame().rename(columns={"crossing_order": "order"})
df_density = density_per_agent.to_frame().rename(columns={"individual_density": "density"})

# Merge on the 'id' index
df_merged = df_order.join(df_density, how="inner")

# df_merged now has columns ['order', 'density'] indexed by 'id'

In [ ]:

plt.figure(figsize=(6,4))

plt.scatter(df_merged["order"], df_merged["mean_density_per_agent"], color="blue")

plt.xlabel("Crossing Order (1 = first to cross)")
plt.ylabel("Mean Density per Agent (1/m^2)")
plt.title("Crossing Order vs. Mean Density")
plt.grid(True)

plt.show()

In [ ]:
from pedpy import plot_voronoi_cells, DENSITY_COL
plot_voronoi_cells(
    voronoi_data=individual,
    traj_data=traj,
    frame=4577,
    walkable_area=bounding_area,
    color_by_column=DENSITY_COL,
    vmin=0,
    vmax=10,
).set_aspect("equal")
plt.show()